In [1]:
import zipfile, os, glob, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Raíz del repo: busca la carpeta data/ en el directorio del notebook o un nivel arriba
_nb_dir = Path(globals()["_dh"][0])
REPO_ROOT = _nb_dir if (_nb_dir / "data").exists() else _nb_dir.parent

zip_path = REPO_ROOT / "data" / "cloud_provider_challenge_dataset_v1.zip"
extract_dir = str(REPO_ROOT / "data" / "extracted")

if not zip_path.exists():
    raise FileNotFoundError(
        f"Poné el zip en: {REPO_ROOT / 'data' / 'cloud_provider_challenge_dataset_v1.zip'}"
    )

os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Descomprimido en: {extract_dir}")


Descomprimido en: c:\Users\fprado\Desktop\data\extracted


In [2]:
data_frames = {}
landing_dir = extract_dir + '/datalake/landing'

for filename in os.listdir(landing_dir):
    if filename.endswith('.csv'):
        df_name = 'df_' + filename.replace('.csv', '')
        data_frames[df_name] = pd.read_csv(os.path.join(landing_dir, filename))

print("Tablas cargadas:", list(data_frames.keys()))

Tablas cargadas: ['df_billing_monthly', 'df_customers_orgs', 'df_marketing_touches', 'df_nps_surveys', 'df_resources', 'df_support_tickets', 'df_users']


In [3]:
stream_dir = extract_dir + '/datalake/landing/usage_events_stream'
jsonl_files = sorted(glob.glob(stream_dir + '/*.jsonl'))
print(f"Archivos .jsonl encontrados: {len(jsonl_files)}")

registros = []
for fp in jsonl_files:
    with open(fp, 'r') as f:
        for linea in f:
            linea = linea.strip()
            if linea:
                registros.append(json.loads(linea))

data_frames['df_usage_events_stream'] = pd.DataFrame(registros)
print(f"Eventos cargados: {len(data_frames['df_usage_events_stream'])}")

Archivos .jsonl encontrados: 120
Eventos cargados: 43200


— Función de exploración

Se creá  una función que hace siempre lo mismo: muestra las primeras filas, los tipos de datos, las estadísticas y los nulos.

In [4]:
def explorar_df(df, nombre):
    print(f"\n{'='*55}")
    print(f"  TABLA: {nombre.upper()}")
    print(f"{'='*55}")
    print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas\n")

    display(df.head(3))
    df.info()

    print("\n--- Estadísticas descriptivas ---")
    display(df.describe())

    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df) * 100).round(2)
    resumen = pd.DataFrame({'Total Nulos': nulos, 'Porcentaje %': nulos_pct})
    resumen = resumen[resumen['Total Nulos'] > 0].sort_values('Porcentaje %', ascending=False)
    print("\n--- Valores nulos ---")
    if len(resumen) > 0:
        display(resumen)
    else:
        print("Sin valores nulos.")

    dup = df.duplicated().sum()
    print(f"\nFilas duplicadas: {dup}")

---

— Exploracion por  tablas

In [5]:
explorar_df(data_frames['df_customers_orgs'], 'customers_orgs')


  TABLA: CUSTOMERS_ORGS
Dimensiones: 80 filas x 11 columnas



,org_id,org_name,industry,hq_region,plan_tier,is_enterprise,signup_date,sales_rep,lifecycle_stage,marketing_source,nps_score
0,org_xaji0y6d,Nimbus Labs 0,Education,sa-east,standard,True,2025-05-26,rep_c,churned,partner,NaN
1,org_pbhsahxt,Nova Tech 1,Education,us-east,standard,False,2025-06-16,rep_d,lead,event,-3.0
2,org_hv3a3zmf,Gamma Data 2,Media,eu-central,pro,True,2025-05-12,rep_c,active,organic,4.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   org_id            80 non-null     object 
 1   org_name          80 non-null     object 
 2   industry          80 non-null     object 
 3   hq_region         80 non-null     object 
 4   plan_tier         80 non-null     object 
 5   is_enterprise     80 non-null     bool   
 6   signup_date       80 non-null     object 
 7   sales_rep         80 non-null     object 
 8   lifecycle_stage   80 non-null     object 
 9   marketing_source  80 non-null     object 
 10  nps_score         69 non-null     float64
dtypes: bool(1), float64(1), object(9)
memory usage: 6.5+ KB

--- Estadísticas descriptivas ---


,nps_score
count,69.000000
mean,18.666667
std,26.071546
min,-38.000000
25%,1.000000
50%,14.000000
75%,34.000000
max,101.000000



--- Valores nulos ---


,Total Nulos,Porcentaje %
nps_score,11,13.75



Filas duplicadas: 0


In [6]:
explorar_df(data_frames['df_users'], 'users')


  TABLA: USERS
Dimensiones: 800 filas x 7 columnas



,user_id,org_id,email,role,active,created_at,last_login
0,user_9tze5r5u,org_3t60rjiw,user_9tze5r5u@example.com,devops,True,2025-07-10,NaN
1,user_qpgb7r3o,org_afeyuhz1,user_qpgb7r3o@example.com,developer,True,2025-07-15,2025-08-10
2,user_cwbfuk9e,org_ykl1cq99,user_cwbfuk9e@example.com,developer,True,2025-06-12,2025-07-26


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     800 non-null    object
 1   org_id      800 non-null    object
 2   email       800 non-null    object
 3   role        800 non-null    object
 4   active      800 non-null    bool  
 5   created_at  800 non-null    object
 6   last_login  661 non-null    object
dtypes: bool(1), object(6)
memory usage: 38.4+ KB

--- Estadísticas descriptivas ---


,user_id,org_id,email,role,active,created_at,last_login
count,800,800,800,800,800,800,661
unique,800,80,800,6,2,100,110
top,user_9tze5r5u,org_a7tz0ync,user_9tze5r5u@example.com,data_engineer,True,2025-05-28,2025-06-26
freq,1,19,1,201,728,16,15



--- Valores nulos ---


,Total Nulos,Porcentaje %
last_login,139,17.38



Filas duplicadas: 0


In [7]:
explorar_df(data_frames['df_resources'], 'resources')


  TABLA: RESOURCES
Dimensiones: 400 filas x 7 columnas



,resource_id,org_id,service,region,created_at,state,tags_json
0,res_eubfn9kr,org_pnsm43d8,compute,sa-east,2025-08-14,running,"[""env:prod""]"
1,res_fvb66h3r,org_i7p5tb94,database,ap-south,2025-06-05,stopped,NaN
2,res_cbrlqmn4,org_d14ve92m,storage,eu-central,2025-08-10,running,"[""env:prod"", ""pii:true""]"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   resource_id  400 non-null    object
 1   org_id       400 non-null    object
 2   service      400 non-null    object
 3   region       400 non-null    object
 4   created_at   400 non-null    object
 5   state        400 non-null    object
 6   tags_json    317 non-null    object
dtypes: object(7)
memory usage: 22.0+ KB

--- Estadísticas descriptivas ---


,resource_id,org_id,service,region,created_at,state,tags_json
count,400,400,400,400,400,400,317
unique,400,80,6,7,106,3,159
top,res_eubfn9kr,org_kdgigatj,compute,ap-northeast,2025-07-25,running,"[""team:ml""]"
freq,1,11,116,72,8,242,17



--- Valores nulos ---


,Total Nulos,Porcentaje %
tags_json,83,20.75



Filas duplicadas: 0


In [8]:
explorar_df(data_frames['df_support_tickets'], 'support_tickets')


  TABLA: SUPPORT_TICKETS
Dimensiones: 1000 filas x 8 columnas



,ticket_id,org_id,category,severity,created_at,resolved_at,csat,sla_breached
0,tkt_zjrumxqw,org_x7eedtjv,performance,low,2025-07-02,2025-07-08,4.0,False
1,tkt_3b4nthzm,org_gv0e38da,security,low,2025-07-21,2025-07-22,NaN,False
2,tkt_32hzs32d,org_9mx2x18h,availability,low,2025-07-03,2025-07-05,3.0,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ticket_id     1000 non-null   object 
 1   org_id        1000 non-null   object 
 2   category      1000 non-null   object 
 3   severity      1000 non-null   object 
 4   created_at    1000 non-null   object 
 5   resolved_at   760 non-null    object 
 6   csat          746 non-null    float64
 7   sla_breached  1000 non-null   bool   
dtypes: bool(1), float64(1), object(6)
memory usage: 55.8+ KB

--- Estadísticas descriptivas ---


,csat
count,746.000000
mean,3.301609
std,1.260200
min,0.000000
25%,3.000000
50%,3.000000
75%,4.000000
max,7.000000



--- Valores nulos ---


,Total Nulos,Porcentaje %
csat,254,25.4
resolved_at,240,24.0



Filas duplicadas: 0


In [9]:
explorar_df(data_frames['df_marketing_touches'], 'marketing_touches')


  TABLA: MARKETING_TOUCHES
Dimensiones: 1500 filas x 7 columnas



,touch_id,org_id,campaign,channel,timestamp,clicked,converted
0,mkt_7322hr5s,org_zbikcidk,security_week,ads,2025-08-26,False,False
1,mkt_osc0whs1,org_5iqvnb4g,webinar_finops,email,2025-07-23,False,False
2,mkt_4eci9rtj,org_okep7y6w,security_week,email,2025-06-21,False,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   touch_id   1500 non-null   object
 1   org_id     1500 non-null   object
 2   campaign   1500 non-null   object
 3   channel    1500 non-null   object
 4   timestamp  1500 non-null   object
 5   clicked    1500 non-null   bool  
 6   converted  1500 non-null   bool  
dtypes: bool(2), object(5)
memory usage: 61.7+ KB

--- Estadísticas descriptivas ---


,touch_id,org_id,campaign,channel,timestamp,clicked,converted
count,1500,1500,1500,1500,1500,1500,1500
unique,1500,80,6,4,120,2,2
top,mkt_7322hr5s,org_pvs7hzio,upgrade_enterprise,event,2025-07-03,False,False
freq,1,32,271,401,23,1047,1356



--- Valores nulos ---
Sin valores nulos.

Filas duplicadas: 0


In [10]:
explorar_df(data_frames['df_nps_surveys'], 'nps_surveys')


  TABLA: NPS_SURVEYS
Dimensiones: 92 filas x 4 columnas



,org_id,survey_date,nps_score,comment
0,org_xaji0y6d,2025-08-22,11.0,Complex billing
1,org_hv3a3zmf,2025-06-13,6.0,Complex billing
2,org_hv3a3zmf,2025-06-18,15.0,Love genAI features


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   org_id       92 non-null     object 
 1   survey_date  92 non-null     object 
 2   nps_score    73 non-null     float64
 3   comment      82 non-null     object 
dtypes: float64(1), object(3)
memory usage: 3.0+ KB

--- Estadísticas descriptivas ---


,nps_score
count,73.000000
mean,26.164384
std,15.893756
min,-16.000000
25%,17.000000
50%,25.000000
75%,36.000000
max,68.000000



--- Valores nulos ---


,Total Nulos,Porcentaje %
nps_score,19,20.65
comment,10,10.87



Filas duplicadas: 0


In [11]:
explorar_df(data_frames['df_billing_monthly'], 'billing_monthly')


  TABLA: BILLING_MONTHLY
Dimensiones: 240 filas x 8 columnas



,invoice_id,org_id,month,subtotal,credits,taxes,currency,exchange_rate_to_usd
0,inv_nym31sk0,org_xaji0y6d,2025-06-01,1133.34,NaN,238.00,USD,0.97684
1,inv_540sejok,org_xaji0y6d,2025-07-01,619.17,NaN,130.03,USD,1.00331
2,inv_ilp4vn2v,org_xaji0y6d,2025-08-01,540.79,NaN,113.57,ARS,0.00160


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   invoice_id            240 non-null    object 
 1   org_id                240 non-null    object 
 2   month                 240 non-null    object 
 3   subtotal              240 non-null    float64
 4   credits               103 non-null    float64
 5   taxes                 240 non-null    float64
 6   currency              240 non-null    object 
 7   exchange_rate_to_usd  240 non-null    float64
dtypes: float64(4), object(4)
memory usage: 15.1+ KB

--- Estadísticas descriptivas ---


,subtotal,credits,taxes,exchange_rate_to_usd
count,240.000000,103.000000,240.000000,240.000000
mean,717.233583,19.099709,171.357458,0.798560
std,556.065046,19.744169,83.252273,0.418672
min,-1671.830000,0.000000,13.180000,0.001330
25%,481.825000,4.490000,110.385000,0.924432
50%,689.865000,12.600000,153.370000,0.980640
75%,1046.585000,24.465000,222.715000,1.040663
max,2237.390000,79.060000,469.850000,1.198080



--- Valores nulos ---


,Total Nulos,Porcentaje %
credits,137,57.08



Filas duplicadas: 0


In [12]:
explorar_df(data_frames['df_usage_events_stream'], 'usage_events_stream')


  TABLA: USAGE_EVENTS_STREAM
Dimensiones: 43200 filas x 13 columnas



,event_id,timestamp,org_id,resource_id,service,region,metric,value,unit,cost_usd_increment,schema_version,carbon_kg,genai_tokens
0,evt_nwlh5boz2guf,2025-08-17T01:55:00Z,org_cvs4f8cg,res_i8kcmm7d,networking,sa-east,requests,None,count,0.9561,2,0.0000,NaN
1,evt_xozb632zko0g,2025-07-16T11:20:00Z,org_rixa11dp,res_nmk7yyz6,database,ap-northeast,requests,None,count,5.6521,1,NaN,NaN
2,evt_h69ugrfzcopy,2025-08-18T18:52:00Z,org_0lzjjege,res_c4t3x1hx,networking,ap-northeast,requests,124.0,count,1.4320,2,0.0248,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43200 entries, 0 to 43199
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   event_id            43200 non-null  object 
 1   timestamp           43200 non-null  object 
 2   org_id              43200 non-null  object 
 3   resource_id         43200 non-null  object 
 4   service             43200 non-null  object 
 5   region              43200 non-null  object 
 6   metric              43200 non-null  object 
 7   value               42323 non-null  object 
 8   unit                41125 non-null  object 
 9   cost_usd_increment  43200 non-null  float64
 10  schema_version      43200 non-null  int64  
 11  carbon_kg           32400 non-null  float64
 12  genai_tokens        3132 non-null   float64
dtypes: float64(3), int64(1), object(9)
memory usage: 4.3+ MB

--- Estadísticas descriptivas ---


,cost_usd_increment,schema_version,carbon_kg,genai_tokens
count,43200.000000,43200.000000,32400.000000,3132.000000
mean,3.412824,1.750000,0.011343,816.845147
std,7.923621,0.433018,0.011450,450.483307
min,-154.460800,1.000000,0.000000,0.000000
25%,0.128500,1.750000,0.000534,476.750000
50%,1.004400,2.000000,0.003386,792.000000
75%,6.117250,2.000000,0.023600,1115.500000
max,317.430800,2.000000,0.032600,2663.000000



--- Valores nulos ---


,Total Nulos,Porcentaje %
genai_tokens,40068,92.75
carbon_kg,10800,25.00
unit,2075,4.80
value,877,2.03



Filas duplicadas: 0


In [13]:
# Tabla comparativa de calidad entre todas las fuentes
resumen_calidad = []

for nombre, df in data_frames.items():
    total_celdas = df.shape[0] * df.shape[1]
    total_nulos = df.isnull().sum().sum()
    resumen_calidad.append({
        'Tabla': nombre.replace('df_', ''),
        'Filas': df.shape[0],
        'Columnas': df.shape[1],
        'Nulos totales': total_nulos,
        'Nulos %': round(total_nulos / total_celdas * 100, 2),
        'Duplicados': df.duplicated().sum()
    })

df_calidad = pd.DataFrame(resumen_calidad).sort_values('Nulos %', ascending=False)
display(df_calidad)

,Tabla,Filas,Columnas,Nulos totales,Nulos %,Duplicados
7,usage_events_stream,43200,13,53820,9.58,0
3,nps_surveys,92,4,29,7.88,0
0,billing_monthly,240,8,137,7.14,0
5,support_tickets,1000,8,494,6.18,0
4,resources,400,7,83,2.96,0
6,users,800,7,139,2.48,0
1,customers_orgs,80,11,11,1.25,0
2,marketing_touches,1500,7,0,0.00,0


Subtotal negativo en billing

un analisis previo mostro que el  describe() de billing mostró un mínimo de -1671.83 por lo que decidimos analizarlo putualmente

In [14]:
df_billing = data_frames['df_billing_monthly']
negativos = df_billing[df_billing['subtotal'] < 0]
print(f"Facturas con subtotal negativo: {len(negativos)}")
display(negativos)

Facturas con subtotal negativo: 13


,invoice_id,org_id,month,subtotal,credits,taxes,currency,exchange_rate_to_usd
23,inv_gx8uwtk7,org_g0fn9xuy,2025-08-01,-1125.54,3.42,236.36,USD,1.00132
74,inv_bmzw7539,org_i7p5tb94,2025-08-01,-849.43,18.20,178.38,USD,1.04212
89,inv_9rvn97y8,org_vyie6ivw,2025-08-01,-910.68,70.98,191.24,USD,0.96732
104,inv_3tlo1i7x,org_x7eedtjv,2025-08-01,-879.97,56.93,184.79,USD,0.95074
113,inv_4dqxin67,org_pac56t4u,2025-08-01,-701.47,2.71,147.31,USD,0.96736
149,inv_0kg6umh4,org_ktakpuxq,2025-08-01,-1355.74,NaN,284.71,ARS,0.00157
150,inv_gblckuhy,org_phr62gds,2025-06-01,-696.82,2.12,146.33,USD,0.99772
152,inv_93pomv7m,org_phr62gds,2025-08-01,-644.77,NaN,135.40,ARS,0.00135
162,inv_eskzadbs,org_g8sbi4q2,2025-06-01,-570.69,NaN,119.84,USD,0.85463
180,inv_mymmhlk1,org_nam148p0,2025-06-01,-538.22,1.21,113.03,USD,1.03612


Nulos en credits

En en analis pudimos ver que credits tiene el 57% de los datos nulos y por lo tanto dedidimos dejar por sentado ese punto ya que lo consideramos importante  maecarlo 

In [15]:
nulos_credits = df_billing['credits'].isnull().sum()
pct = nulos_credits / len(df_billing) * 100
print(f"Nulos en credits: {nulos_credits} ({pct:.2f}%)")

Nulos en credits: 137 (57.08%)


Carbon_kg según schema_version

En el  análisis exploratirio tambien se  mostró que el  25% de los nulos estas en Carbon_kg.  por lo que decidimos analizar   si esos nulos corresponden exactamente a los eventos con schema_version = 1 o schema_version=2

In [16]:
df_usage = data_frames['df_usage_events_stream']
resultado = df_usage.groupby('schema_version')['carbon_kg'].apply(lambda x: x.isnull().sum())
print("Nulos en carbon_kg por schema_version:")
print(resultado)

Nulos en carbon_kg por schema_version:
schema_version
1    10800
2        0
Name: carbon_kg, dtype: int64
